In [3]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.lines import Line2D
import plotly.express as px
import pandas as pd
import plotly.graph_objects as go
import plotly.colors as pc
import os

In [ ]:
#weight tensors for layer 1
l1_forget_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_forget_gate_ih.pt')
l1_forget_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_forget_gate_hh.pt')
l1_input_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_input_gate_ih.pt')
l1_input_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_input_gate_hh.pt')
l1_cell_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_cell_gate_ih.pt')
l1_cell_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_cell_gate_hh.pt')
l1_output_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_output_gate_ih.pt')
l1_output_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer1_output_gate_hh.pt')

#weight tensors for layer 2
l2_forget_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_forget_gate_ih.pt')
l2_forget_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_forget_gate_hh.pt')
l2_input_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_input_gate_ih.pt')
l2_input_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_input_gate_hh.pt')
l2_cell_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_cell_gate_ih.pt')
l2_cell_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_cell_gate_hh.pt')
l2_output_ih = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_output_gate_ih.pt')
l2_output_hh = torch.load('/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights/adam_training/layer2_output_gate_hh.pt')

In [5]:
num_checkpoints = l1_forget_ih.shape[0]

In [6]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.spatial import ConvexHull
import seaborn as sns
import pandas as pd

def plot_3d_eigenvalue_evolution(eigenvalues_list, gate_names, epochs, figsize=(16, 14), top_n=10):
    """
    Plot 3D Adrian diagrams with enhanced visualization of eigenvalue shapes at each epoch
    
    Parameters:
    - eigenvalues_list: List of lists of eigenvalues per gate across epochs
    - gate_names: List of gate names
    - epochs: Array of epoch numbers
    - figsize: Figure size
    - top_n: Number of top eigenvalues to plot (selected by magnitude)
    """
    plt.ion()

    num_gates = len(eigenvalues_list)
    
    # Create a figure with 3D subplots
    fig = plt.figure(figsize=figsize)
    
    for gate_idx, gate_name in enumerate(gate_names):
        # Create a 3D subplot
        ax = fig.add_subplot(2, 2, gate_idx + 1, projection='3d')
        
        # Prepare colormap for epochs
        cmap = plt.cm.viridis
        norm = plt.Normalize(min(epochs), max(epochs))
        
        # For each epoch, plot the eigenvalues
        for epoch_idx, epoch in enumerate(epochs):
            eig_vals = eigenvalues_list[gate_idx][epoch_idx]
            
            # Select top_n eigenvalues by magnitude (in terms of distance to the origin in the complex plan)
            magnitudes = np.abs(eig_vals)#sqrt(real^2 + imag^2)
            top_indices = np.argsort(magnitudes)[-top_n:]  # Get indices of top_n largest magnitudes
            
            # Select only the top eigenvalues
            top_eig_vals = eig_vals[top_indices]
            
            real_parts = np.real(top_eig_vals)
            imag_parts = np.imag(top_eig_vals)
            # Create a time coordinate for all eigenvalues at this epoch
            time_coords = np.full_like(real_parts, epoch)
            
            # Plot eigenvalues as points in 3D space
            scatter = ax.scatter(
                real_parts, 
                imag_parts, 
                time_coords,
                c=[cmap(norm(epoch))] * len(top_eig_vals),
                alpha=0.8,
                s=60,  # Larger points for better visibility
                label=f'Epoch {epoch}'
            )
            
            # Enhanced visualization of the shape:
            # Method 1: Connect points in order of angle for a clean polygon
            angles = np.angle(top_eig_vals)
            sorted_indices = np.argsort(angles)
            
            # Create arrays for the sorted eigenvalues to make a closed polygon
            sorted_real = real_parts[sorted_indices]
            sorted_imag = imag_parts[sorted_indices]
            
            # Add the first point again to close the loop
            sorted_real = np.append(sorted_real, sorted_real[0])
            sorted_imag = np.append(sorted_imag, sorted_imag[0])
            time_array = np.full_like(sorted_real, epoch)
            
            # Draw the polygon connecting eigenvalues
            ax.plot(
                sorted_real,
                sorted_imag,
                time_array,
                color=cmap(norm(epoch)),
                alpha=0.6,
                linewidth=2.5
            )
            
            # Method 2: Fill the polygon with semi-transparent surface for better shape visualization
            # Only do this if we have at least 3 eigenvalues (required for a polygon)
            if len(top_eig_vals) >= 3:
                # Create a polygon patch
                polygon = np.column_stack((sorted_real[:-1], sorted_imag[:-1]))  # Remove duplicate point
                
                try:
                    # Try to create a convex hull for more stable surface plotting
                    hull = ConvexHull(polygon)
                    # Create a surface using the hull vertices
                    for simplex in hull.simplices:
                        x = polygon[simplex, 0]
                        y = polygon[simplex, 1]
                        z = np.full_like(x, epoch)
                        ax.plot_trisurf(x, y, z, color=cmap(norm(epoch)), alpha=0.5)
                except:
                    # Fall back to simple polygon if convex hull fails
                    ax.plot_trisurf(
                        sorted_real[:-1],  # Exclude the duplicate point
                        sorted_imag[:-1], 
                        np.full_like(sorted_real[:-1], epoch),
                        color=cmap(norm(epoch)), 
                        alpha=0.2
                    )
        
        # Add unit circle at each time step
        theta = np.linspace(0, 2*np.pi, 100)
        for epoch in epochs:
            x = np.cos(theta)
            y = np.sin(theta)
            z = np.full_like(theta, epoch)
            ax.plot(x, y, z, 'r--', alpha=0.5, linewidth=1.5)
        
        # Set labels and title
        ax.set_xlabel('Real Part', fontsize=12)
        ax.set_ylabel('Imaginary Part', fontsize=12)
        ax.set_zlabel('Epoch', fontsize=12)
        ax.set_title(f'{gate_name} - Eigenvalue Shape Evolution', fontsize=14, fontweight='bold')
        
        # Set equal aspect ratio for x and y to ensure circle looks like a circle
        # This makes the curvature more apparent
        ax.set_box_aspect([1, 1, 1.5])
        
        # Add grid for better visualization
        ax.grid(True, alpha=0.3)
        
        # Add a colorbar to indicate epochs
        sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
        sm.set_array([])
        cbar = plt.colorbar(sm, ax=ax, label='Epoch')
    
    plt.tight_layout()

    # Enable rotation and view adjustment
    for ax in fig.get_axes():
        if hasattr(ax, 'view_init'):  # Check if it's a 3D axis
            ax.view_init(elev=0, azim=45)  # Set initial viewing angle
        
    return fig

In [7]:
def load_eigenvalues_from_tensor_csv(file_path):
    """
    Load a CSV containing a 3D tensor (num_checkpoints, hidden_size, hidden_size),
    and compute eigenvalues for each checkpoint.

    Returns:
        List of eigenvalues (one entry per checkpoint)
    """
    tensor = torch.load(file_path)  # Should be shape (num_checkpoints, H, H)
    if isinstance(tensor, torch.Tensor):
        tensor = tensor.numpy()  # Convert to NumPy array if it's a tensor

    num_checkpoints = tensor.shape[0]

    eigenvalues_list = [np.linalg.eigvals(tensor[i]) for i in range(num_checkpoints)]
    return eigenvalues_list


In [8]:
def load_all_gates_eigenvalues(matrix_dir, layer_nb, gates, connection_type):
    """
    Load eigenvalues for all gates from their respective CSVs.
    
    Args:
        matrix_dir (str): Directory path where CSVs are located.
        gate_names (list): List of gate names.
        connection_type (str): Either 'hh' or 'ih'.

    Returns:
        List of list of eigenvalues: shape (num_gates, num_checkpoints)
    """
    eigenvalues_all_gates = []

    for gate in gates:
        file_name = f"layer{layer_nb}_{gate}_gate_{connection_type}.pt"  # Adapt this if filenames differ
        file_path = os.path.join(matrix_dir, file_name)
        eigs = load_eigenvalues_from_tensor_csv(file_path)
        eigenvalues_all_gates.append(eigs)

    return eigenvalues_all_gates


In [ ]:
gate_names = ['input', 'forget', 'output', 'cell']
matrix_dir = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights"  # change this
epochs = np.arange(num_checkpoints)  # or however many checkpoints you have

# Load input-to-hidden and hidden-to-hidden
eig_ih = load_all_gates_eigenvalues(matrix_dir, 1, gate_names, connection_type='ih')
eig_hh = load_all_gates_eigenvalues(matrix_dir, 1, gate_names, connection_type='hh')

# Plot them
fig1 = plot_3d_eigenvalue_evolution(eig_ih, gate_names, epochs, top_n=10)
plt.show()

fig2 = plot_3d_eigenvalue_evolution(eig_hh, gate_names, epochs, top_n=10)
plt.show()

In [ ]:
gate_names = ['input', 'forget', 'output', 'cell']
matrix_dir = "/scratch2/mrenaudin/colorlessgreenRNNs/checkpoints/lstm_adam/weights"  # change this
epochs = np.arange(num_checkpoints)  # or however many checkpoints you have

# Load input-to-hidden and hidden-to-hidden
eig_ih = load_all_gates_eigenvalues(matrix_dir, 2, gate_names, connection_type='ih')
eig_hh = load_all_gates_eigenvalues(matrix_dir, 2, gate_names, connection_type='hh')

# Plot them
fig1 = plot_3d_eigenvalue_evolution(eig_ih, gate_names, epochs, top_n=10)
plt.show()

fig2 = plot_3d_eigenvalue_evolution(eig_hh, gate_names, epochs, top_n=10)
plt.show()